# Avaloka analysis

**Goal:** average is_late by customer_state

Reproducible notebook generated by Avaloka mission `cf9254428d36`.

In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', 100)

## Load

In [ ]:
DATA_SOURCE = '/mnt/e/Avloka/data/prepared/orders_analysis.csv'


# The strings read as missing, per column, exactly as the analysis read them.
MISSING_STRINGS = {
    **dict.fromkeys(['customer_state', 'seller_state', 'main_category', 'payment_type', 'purchase_weekday', 'purchase_month'], ['']),
    **dict.fromkeys(['same_state', 'n_items', 'n_sellers', 'items_value', 'freight_value', 'freight_share', 'total_weight_g', 'max_installments', 'purchase_hour', 'promised_days', 'days_late', 'review_score', 'is_late'], ['', '#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan', '1.#IND', '1.#QNAN', '<NA>', 'N/A', 'NA', 'NULL', 'NaN', 'None', 'n/a', 'nan', 'null']),
}


def load() -> pd.DataFrame:
    df = pd.read_csv(
        DATA_SOURCE, sep=',', encoding='utf-8', keep_default_na=False, na_values=MISSING_STRINGS)
    return df.dropna(axis=1, how="all")  # all-empty columns, as the analysis dropped them


df = load()
print(df.shape)
df.head()

## Profile

In [ ]:
df.describe(include='all').T

In [ ]:
df.isna().mean().sort_values(ascending=False).head(20)  # missingness

## Transform (Avaloka-selected)

In [ ]:
def transform(df):
    df = df.drop_duplicates().reset_index(drop=True)  # exact duplicates
    for _c in ['customer_state', 'seller_state', 'main_category', 'payment_type', 'purchase_weekday', 'purchase_month']:
        df[_c] = df[_c].astype('string').str.strip()
    return df

df = transform(df)
print(df.shape)

## Distributions

In [ ]:
num = df.select_dtypes('number')
if num.shape[1]:
    num.hist(figsize=(12, 8))

## Notes

- Working rows analysed: 96,470
- See `validation_report.json` for leakage / stability checks.
- See `assumptions.yaml` for every judgement call.